In [119]:
# Importando bibliotecas e informações

import pandas as pd

url = 'https://dados.mobilidade.rio/gps/sppo?dataInicial=2026-03-09+08:04:00&dataFinal=2026-03-09+08:09:00'

df = pd.read_json(url)

## Tratando df

In [120]:
# Tratando os valores de latitude e longitude

df['latitude'] = df['latitude'].str.replace(',', '.', regex=False).astype(float)
df['longitude'] = df['longitude'].str.replace(',', '.', regex=False).astype(float)

In [121]:
# Transformando datahora (UNIX) em data (Gregoriana)

print(df['datahora'].dtype)
# Convert 'datahora' from Unix timestamp (milliseconds) to datetime objects
df['datahora_gregoriana'] = pd.to_datetime(df['datahora'], unit='ms')

# Convert 'datahoraenvio' from Unix timestamp (milliseconds) to datetime objects
df['datahoraenvio_gregoriana'] = pd.to_datetime(df['datahoraenvio'], unit='ms')

# Convert 'datahoraservidor' from Unix timestamp (milliseconds) to datetime objects
df['datahoraservidor_gregoriana'] = pd.to_datetime(df['datahoraservidor'], unit='ms')


int64


In [122]:
# Excluindo colunas

df = df.drop(columns=['datahora','datahoraenvio', 'datahoraservidor'])
df

,ordem,latitude,longitude,velocidade,linha,datahora_gregoriana,datahoraenvio_gregoriana,datahoraservidor_gregoriana
0,B31120,-22.90393,-43.21523,68,485,2026-03-09 11:03:46,2026-03-09 11:04:00,2026-03-09 11:04:00
1,B31003,-22.85494,-43.25965,22,497,2026-03-09 11:03:48,2026-03-09 11:04:00,2026-03-09 11:04:00
2,B31004,-22.83700,-43.28645,0,GARAGEM,2026-03-09 11:03:50,2026-03-09 11:04:00,2026-03-09 11:04:00
3,B31026,-22.93739,-43.17104,48,SPA483,2026-03-09 11:03:49,2026-03-09 11:04:00,2026-03-09 11:04:00
4,B31009,-22.97548,-43.19203,37,SPA483,2026-03-09 11:03:52,2026-03-09 11:04:00,2026-03-09 11:04:00
...,...,...,...,...,...,...,...,...
41787,D12082,-22.83694,-43.56277,0,830,2026-03-09 11:08:56,2026-03-09 11:09:00,2026-03-09 11:09:05
41788,D12058,-22.90187,-43.55960,0,850,2026-03-09 11:08:57,2026-03-09 11:09:00,2026-03-09 11:09:05
41789,D12064,-22.85643,-43.54161,24,895,2026-03-09 11:08:57,2026-03-09 11:09:00,2026-03-09 11:09:05
41790,D12061,-22.85717,-43.54158,12,850,2026-03-09 11:08:57,2026-03-09 11:09:00,2026-03-09 11:09:05


## Filtro por linha

In [128]:
# Escolha a linha que deseja analisar

selected_linha = input('')
df_filtered_local = df[df['linha'] == selected_linha]

df_filtered_local.head(10)

630


,ordem,latitude,longitude,velocidade,linha,datahora_gregoriana,datahoraenvio_gregoriana,datahoraservidor_gregoriana
1388,B31105,-22.84209,-43.27222,0,630,2026-03-09 11:03:57,2026-03-09 11:04:10,2026-03-09 11:04:30
1464,B31064,-22.91613,-43.23712,5,630,2026-03-09 11:04:08,2026-03-09 11:04:11,2026-03-09 11:04:31
1933,B27160,-22.91910,-43.24075,3,630,2026-03-09 11:04:01,2026-03-09 11:04:15,2026-03-09 11:04:42
2140,B27248,-22.85254,-43.26188,12,630,2026-03-09 11:04:04,2026-03-09 11:04:16,2026-03-09 11:04:42
2149,B27095,-22.90090,-43.24108,27,630,2026-03-09 11:04:08,2026-03-09 11:04:16,2026-03-09 11:04:42
2181,B27242,-22.91886,-43.25393,0,630,2026-03-09 11:04:13,2026-03-09 11:04:16,2026-03-09 11:04:42
2186,B27234,-22.89156,-43.24064,0,630,2026-03-09 11:04:13,2026-03-09 11:04:16,2026-03-09 11:04:42
2489,B31005,-22.83571,-43.27694,37,630,2026-03-09 11:04:13,2026-03-09 11:04:20,2026-03-09 11:04:31
2527,B31156,-22.87485,-43.25756,37,630,2026-03-09 11:04:11,2026-03-09 11:04:20,2026-03-09 11:04:31
4963,B27160,-22.91758,-43.24165,29,630,2026-03-09 11:04:32,2026-03-09 11:04:37,2026-03-09 11:04:42


## Gerando as rotas de determinada linha

In [125]:
get_ipython().system('pip install folium')

In [126]:
import folium


In [129]:
center_lat = df_filtered_local['latitude'].mean()
center_lon = df_filtered_local['longitude'].mean()

mapa = folium.Map(location=[center_lat, center_lon], zoom_start=13)

folium.TileLayer('OpenStreetMap').add_to(mapa)

In [130]:
for index, row in df_filtered_local.iterrows():
    folium.Marker([row['latitude'], row['longitude']], tooltip=f"Bus: {row['ordem']}, Velocidade: {row['velocidade']}").add_to(mapa)

In [131]:
unique_onibus = df_filtered_local['ordem'].unique()

for bus_id in unique_onibus:
    df_bus = df_filtered_local[df_filtered_local['ordem'] == bus_id].sort_values(by='datahora_gregoriana')

    if not df_bus.empty:
        trajetória = df_bus[['latitude', 'longitude']].values.tolist()

        if len(trajetória) > 1:

            folium.PolyLine(trajetória, color='blue', weight=2.5, opacity=0.7).add_to(mapa)


            start_point = trajetória[0]
            folium.Marker(
                start_point,
                tooltip=f"Bus ID: {bus_id}\nStart: {df_bus['datahora_gregoriana'].iloc[0].strftime('%Y-%m-%d %H:%M:%S')}",
                icon=folium.Icon(color='green', icon='play', prefix='fa')
            ).add_to(mapa)


            end_point = trajetória[-1]
            folium.Marker(
                end_point,
                tooltip=f"Bus ID: {bus_id}\nEnd: {df_bus['datahora_gregoriana'].iloc[-1].strftime('%Y-%m-%d %H:%M:%S')}",
                icon=folium.Icon(color='red', icon='stop', prefix='fa')
            ).add_to(mapa)

mapa

## Estatísticas dos Ônibus

In [132]:
df_bus_summary['total_distancia_km'] = df_bus_summary['total_distancia_km'].round(2)
df_bus_summary['velocidade_media_km_h'] = df_bus_summary['velocidade_media_km_h'].round(2)


print("Resumo por ônibus (Distância Total em KM e Velocidade Média em KM/H):")
print(f"Linha: {selected_linha}")
print(df_bus_summary[['ordem', 'total_distancia_km', 'velocidade_media_km_h']].to_string(index=False))

Resumo por ônibus (Distância Total em KM e Velocidade Média em KM/H):
Linha: 630
 ordem  total_distancia_km  velocidade_media_km_h
D33207                1.91                  24.12
D33211                1.41                  17.68
D33227                0.68                   9.80
D33268                1.63                  21.07
D33310                0.01                   0.10
